<a href="https://colab.research.google.com/github/arcctg/kpi-ml-lab6/blob/main/01_applicant_admission_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Applicant Admission Prediction

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 1. Data Generation

In [ ]:
np.random.seed(42)

N = 1500
PRIVILEGED_RATIO = 0.13
N_ANOMALIES = 50

math_scores = np.clip(np.random.normal(155, 20, N), 100, 200).astype(int)
english_scores = np.clip(np.random.normal(155, 20, N), 100, 200).astype(int)
ukrainian_scores = np.clip(np.random.normal(155, 20, N), 100, 200).astype(int)

privileged = np.zeros(N, dtype=int)
privileged_indices = np.random.choice(N, size=int(N * PRIVILEGED_RATIO), replace=False)
privileged[privileged_indices] = 1

anomaly_indices = np.random.choice(N, size=N_ANOMALIES, replace=False)
for i, idx in enumerate(anomaly_indices):
    anomaly_type = i % 5
    if anomaly_type == 0:
        math_scores[idx] = np.random.randint(190, 201)
        english_scores[idx] = np.random.randint(180, 201)
        ukrainian_scores[idx] = np.random.randint(100, 115)
    elif anomaly_type == 1:
        math_scores[idx] = np.random.randint(100, 115)
        english_scores[idx] = np.random.randint(100, 115)
        ukrainian_scores[idx] = np.random.randint(100, 115)
    elif anomaly_type == 2:
        math_scores[idx] = np.random.randint(100, 130)
        english_scores[idx] = np.random.randint(190, 201)
        ukrainian_scores[idx] = np.random.randint(170, 201)
    elif anomaly_type == 3:
        math_scores[idx] = 200
        english_scores[idx] = 200
        ukrainian_scores[idx] = 200
    elif anomaly_type == 4:
        math_scores[idx] = np.random.randint(180, 201)
        english_scores[idx] = np.random.randint(100, 115)
        ukrainian_scores[idx] = np.random.randint(180, 201)

rating = 0.4 * math_scores + 0.3 * english_scores + 0.3 * ukrainian_scores

df = pd.DataFrame({
    'id': range(1, N + 1),
    'math': math_scores,
    'english': english_scores,
    'ukrainian': ukrainian_scores,
    'privileged': privileged,
    'rating': rating
})

print(f"Generated {N} applicants")
print(f"Privileged: {privileged.sum()} ({privileged.sum()/N*100:.1f}%)")
print(f"Anomalous cases injected: {N_ANOMALIES}")

In [ ]:
def determine_admission(df, total_seats=350, privileged_quota=35):
    admitted = np.zeros(len(df), dtype=int)

    priv_mask = df['privileged'] == 1
    non_priv_mask = df['privileged'] == 0

    priv_eligible = df[priv_mask &
                       (df['math'] >= 120) &
                       (df['english'] >= 120) &
                       (df['ukrainian'] >= 120) &
                       (df['rating'] >= 144)].copy()

    non_priv_eligible = df[non_priv_mask &
                           (df['math'] >= 140) &
                           (df['english'] >= 120) &
                           (df['ukrainian'] >= 120) &
                           (df['rating'] >= 160)].copy()

    priv_eligible = priv_eligible.sort_values('rating', ascending=False)
    non_priv_eligible = non_priv_eligible.sort_values('rating', ascending=False)

    priv_admitted = priv_eligible.head(privileged_quota)
    n_priv_admitted = len(priv_admitted)

    remaining_seats = total_seats - n_priv_admitted
    non_priv_admitted = non_priv_eligible.head(remaining_seats)

    admitted_indices = list(priv_admitted.index) + list(non_priv_admitted.index)
    admitted[admitted_indices] = 1

    return admitted

df['admitted'] = determine_admission(df)

os.makedirs('data', exist_ok=True)
df.to_csv('data/applicants.csv', index=False)

print(f"Dataset shape: {df.shape}")
print(f"Admitted: {df['admitted'].sum()}")
print(f"Rejected: {(df['admitted'] == 0).sum()}")
print(f"Privileged total: {df['privileged'].sum()}")
print(f"Privileged admitted: {df[(df['privileged'] == 1) & (df['admitted'] == 1)].shape[0]}")
print(f"Non-privileged admitted: {df[(df['privileged'] == 0) & (df['admitted'] == 1)].shape[0]}")

## 2. Exploratory Data Analysis

In [ ]:
df.head(10)

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, color in zip(axes, ['math', 'english', 'ukrainian'],
                           ['#2196F3', '#4CAF50', '#FF9800']):
    ax.hist(df[col], bins=30, color=color, edgecolor='white', alpha=0.8)
    ax.set_title(f'{col.capitalize()} Score Distribution')
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.axvline(x=df[col].mean(), color='red', linestyle='--',
               label=f'Mean: {df[col].mean():.1f}')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(df[df['admitted'] == 1]['rating'], bins=30, alpha=0.7,
        color='#4CAF50', label='Admitted', edgecolor='white')
ax.hist(df[df['admitted'] == 0]['rating'], bins=30, alpha=0.7,
        color='#F44336', label='Rejected', edgecolor='white')
ax.set_title('Rating Distribution: Admitted vs Rejected')
ax.set_xlabel('Rating')
ax.set_ylabel('Count')
ax.axvline(x=160, color='orange', linestyle='--', linewidth=2,
           label='Min rating (non-privileged): 160')
ax.axvline(x=144, color='purple', linestyle='--', linewidth=2,
           label='Min rating (privileged): 144')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

priv_counts = df['privileged'].value_counts()
axes[0].pie(priv_counts, labels=['Non-privileged', 'Privileged'],
            autopct='%1.1f%%', colors=['#2196F3', '#FF9800'], startangle=90)
axes[0].set_title('Overall: Privileged vs Non-privileged')

admitted_priv = df[df['admitted'] == 1]['privileged'].value_counts()
axes[1].pie(admitted_priv, labels=['Non-privileged', 'Privileged'],
            autopct='%1.1f%%', colors=['#2196F3', '#FF9800'], startangle=90)
axes[1].set_title('Admitted: Privileged vs Non-privileged')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

correlation = df[['math', 'english', 'ukrainian', 'privileged',
                   'rating', 'admitted']].corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', ax=ax)
ax.set_title('Correlation Matrix')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(df['rating'], df['math'], c=df['admitted'],
                     cmap='RdYlGn', alpha=0.5, edgecolors='none')
ax.set_xlabel('Rating')
ax.set_ylabel('Math Score')
ax.set_title('Rating vs Math Score (colored by admission)')
ax.axvline(x=160, color='orange', linestyle='--', alpha=0.7,
           label='Min rating: 160')
ax.axhline(y=140, color='red', linestyle='--', alpha=0.7,
           label='Min math: 140')
plt.colorbar(scatter, label='Admitted')
ax.legend()

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
features = ['math', 'english', 'ukrainian', 'privileged']
target = 'admitted'

X = df[features].values
y = df[target].values

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Feature columns: {features}")

In [ ]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

print("Before scaling:")
print(f"  Min: {X.min(axis=0)}")
print(f"  Max: {X.max(axis=0)}")
print(f"\nAfter scaling:")
print(f"  Min: {X_scaled.min(axis=0)}")
print(f"  Max: {X_scaled.max(axis=0)}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining class distribution:")
print(f"  Admitted: {y_train.sum()} ({y_train.sum()/len(y_train)*100:.1f}%)")
print(f"  Rejected: {(y_train == 0).sum()} ({(y_train == 0).sum()/len(y_train)*100:.1f}%)")
print(f"\nTest class distribution:")
print(f"  Admitted: {y_test.sum()} ({y_test.sum()/len(y_test)*100:.1f}%)")
print(f"  Rejected: {(y_test == 0).sum()} ({(y_test == 0).sum()/len(y_test)*100:.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, data, title in zip(axes, [y_train, y_test], ['Training Set', 'Test Set']):
    counts = [np.sum(data == 0), np.sum(data == 1)]
    bars = ax.bar(['Rejected', 'Admitted'], counts,
                  color=['#F44336', '#4CAF50'], edgecolor='white')
    ax.set_title(f'{title} Class Distribution')
    ax.set_ylabel('Count')
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5,
                f'{count}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Neural Network Models

In [ ]:
def build_model(architecture, optimizer='adam', input_dim=4):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for units, activation in architecture:
        model.add(layers.Dense(units, activation=activation))

    model.add(layers.Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

def train_model(model, X_train, y_train, X_test, y_test,
                epochs=100, batch_size=32, verbose=0):
    start_time = time.time()
    history = model.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_test, y_test),
        verbose=verbose
    )
    training_time = time.time() - start_time
    return history, training_time

def plot_training_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_title(f'{title} - Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['accuracy'], label='Train Accuracy')
    axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[1].set_title(f'{title} - Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

model_results = {}

### Model 1: Minimal Network (1 hidden layer, 8 neurons)

In [ ]:
tf.random.set_seed(42)

model1_arch = [(8, 'sigmoid')]
model1 = build_model(model1_arch, optimizer='adam')
model1.summary()

In [ ]:
history1, time1 = train_model(model1, X_train, y_train, X_test, y_test,
                               epochs=100, batch_size=32)

loss1, acc1 = model1.evaluate(X_test, y_test, verbose=0)
print(f"Model 1 - Test Loss: {loss1:.4f}, Test Accuracy: {acc1:.4f}")
print(f"Training time: {time1:.2f}s")

model_results['Model 1 (8-sigmoid)'] = {
    'accuracy': acc1, 'loss': loss1, 'time': time1,
    'params': model1.count_params()
}

In [ ]:
plot_training_history(history1, 'Model 1: Minimal (8-sigmoid)')